## Dynamic views — the older fine-grained alternative

Before column masks and row filters became first-class, the bank's pattern was a **dynamic view** — a regular view whose `SELECT` uses `CASE WHEN is_account_group_member(...)` to mask columns *and* filter rows in one object.

```sql
CREATE OR REPLACE VIEW fintech_dev.gold.customers_secure AS
SELECT customer_id,
       CASE WHEN is_account_group_member('compliance') THEN email ELSE 'REDACTED' END AS email,
       city, country, created_at
FROM   fintech_dev.silver.customers
WHERE  CASE WHEN is_account_group_member('compliance_in') THEN country = 'IN'
            WHEN is_account_group_member('compliance')    THEN TRUE
            ELSE FALSE END;
```

**When to still use dynamic views:** very simple cases, or compatibility with non-UC clients that don't understand masks.

**Why masks + row filters (and ABAC) are preferred now:**

- They apply to the **base table**, so *every* reader and *every* query path gets them — no "forgot to use the secure view" failure mode.
- They compose with ordinary grants instead of forcing you to re-grant a separate view.
- ABAC scales them across many tables via tags.

**The exam framing:** dynamic views are the **legacy** fine-grained control; column masks + row filters + ABAC are the **modern** answer. If a question contrasts them, pick masks/filters unless it specifically calls for the old view-based approach or a non-UC client.
